# Quick Start

[Download this notebook](quick_start.ipynb)

Run this notebook from top to bottom after [installing `msmu`](https://bertis-informatics.github.io/msmu/installation/).
It uses a small Sage label-free dataset (PXD012986, six samples in groups G1 and G2)
and its SDRF metadata. Input URLs are pinned to a repository revision so the example
does not change when the development branch changes. Network access is required.

The workflow reads search results, filters identifications, normalizes peptide
intensities, infers protein groups, plots their intensities, and saves/reloads the result.
For experiment-specific options see [DDA-LFQ](https://bertis-informatics.github.io/msmu/tutorials/dda-lfq/), [DDA-TMT](https://bertis-informatics.github.io/msmu/tutorials/dda-tmt/),
[DIA-LFQ](https://bertis-informatics.github.io/msmu/tutorials/dia-lfq/), and [DE Analysis](https://bertis-informatics.github.io/msmu/tutorials/dea/).


In [1]:
from pathlib import Path

import msmu as mm
import numpy as np
import pandas as pd

print("msmu", mm.__version__)


## 1. Read search results and sample metadata

Sage identification and quantification are separate files. For label-free DDA,
`psm` holds identifications and `peptide` holds intensities. TMT instead requires
its reporter-intensity `tmt.tsv`; a directory alone is not a valid input.

Attach the original SDRF, then project its sample annotations onto observations.


In [2]:
base = "https://raw.githubusercontent.com/bertis-informatics/msmu/599a92d65bf28b6223d59c96d691a23bb87501d9/data/sage_lfq"
mdata = mm.read_sage(
    identification_file=f"{base}/sage/results.sage.tsv",
    quantification_file=f"{base}/sage/lfq.tsv",
    label="label_free",
)
mdata = mm.pp.attach_sdrf(mdata, f"{base}/meta.sdrf.tsv")
mdata = mm.pp.apply_sdrf_to_obs(mdata)

assert set(mdata["peptide"].obs["factor value[condition]"]) == {"G1", "G2"}
mdata.obs[["source name", "factor value[condition]"]]


INFO - Reading SAGE Identification data: 1 file(s)


INFO - Reading SAGE Quantification data: 1 file(s)


INFO - Validating SDRF metadata for https://raw.githubusercontent.com/bertis-informatics/msmu/599a92d65bf28b6223d59c96d691a23bb87501d9/data/sage_lfq/meta.sdrf.tsv.


INFO - SDRF validation succeeded for https://raw.githubusercontent.com/bertis-informatics/msmu/599a92d65bf28b6223d59c96d691a23bb87501d9/data/sage_lfq/meta.sdrf.tsv.


,source name,factor value[condition]
QExHF04026,G1-1,G1
QExHF04028,G2-1,G2
QExHF04036,G1-2,G1
QExHF04038,G2-2,G2
QExHF04046,G1-3,G1
QExHF04048,G2-3,G2


## 2. Filter and summarize identifications

Keep PSMs below a 1% identification q-value, then summarize their peptide evidence
and filter at peptide level. These are identification q-values, not DE q-values.
Always retain the returned MuData when chaining these preprocessing operations.


In [3]:
mdata = mm.pp.add_filter(mdata, modality="psm", column="q_value", keep="lt", value=0.01)
mdata = mm.pp.apply_filter(mdata, modality="psm", on="var")
mdata = mm.pp.to_peptide(mdata)
mdata = mm.pp.add_filter(mdata, modality="peptide", column="q_value", keep="lt", value=0.01)
mdata = mm.pp.apply_filter(mdata, modality="peptide", on="var")


INFO - Applying var filters for psm: ['q_value_lt_0.01']


INFO - Peptide-level identifications: 3683 (3664 at 1% FDR)


INFO - Using existing peptide quantification data.


INFO - Applying var filters for peptide: ['q_value_lt_0.01']


## 3. Normalize peptide intensities and infer proteins

Log2-transform once before median normalization. Protein inference supplies
`protein_group` and `peptide_type`; the protein rollup uses the three unique peptides
with highest median intensity per group. Other aggregation choices are described in
[Summarization](https://bertis-informatics.github.io/msmu/how-it-works/summarization/).


In [4]:
mdata = mm.pp.log2_transform(mdata, modality="peptide")
mdata = mm.pp.normalise(mdata, modality="peptide", method="median")
mdata = mm.pp.infer_protein(mdata)
mdata = mm.pp.to_protein(mdata, top_n=3, rank_method="median_intensity")
mdata = mm.pp.add_filter(mdata, modality="protein", column="q_value", keep="lt", value=0.01)
mdata = mm.pp.apply_filter(mdata, modality="protein", on="var")

assert mdata["protein"].n_obs == 6
assert mdata["protein"].n_vars > 0
assert np.isfinite(mdata["protein"].X).any()
mdata["protein"]


INFO - Starting protein inference


INFO - Initial proteins: 3721


INFO - Removed indistinguishable: 1624


INFO - Removed subsettable: 559


INFO - Removed subsumable: 2


INFO - Total protein groups: 1536


INFO - Applying var filters for peptide: ['q_value_lt_0.01', 'peptide_type_eq_unique']


INFO - Protein-level identifications: 1501 (1475 at 1% FDR)


INFO - Applying var filters for protein: ['q_value_lt_0.01']


AnnData object with n_obs × n_vars = 6 × 1475
    obs: 'source name', 'characteristics[organism]', 'characteristics[organism part]', 'characteristics[cell line]', 'characteristics[cell type]', 'characteristics[cellosaurus accession]', 'characteristics[cellosaurus name]', 'characteristics[disease]', 'characteristics[biological replicate]', 'assay name', 'technology type', 'comment[proteomexchange accession number]', 'comment[proteomics data acquisition method]', 'comment[fraction identifier]', 'comment[technical replicate]', 'comment[label]', 'comment[instrument]', 'comment[cleavage agent details]', 'factor value[condition]'
    var: 'count_psm', 'count_stripped_peptide', 'PEP', 'q_value'
    uns: 'level', 'decoy', 'filter', 'decoy_filter'
    varm: 'filter'
    layers: None (.X)

## 4. Plot and export

Plotly returns an interactive figure. The HTML export embeds its JavaScript and
can be opened without a notebook server or Chrome image-export setup.
Rerunning this notebook overwrites the files in the output directory.


In [5]:
output_dir = Path("msmu_quick_start_output")
output_dir.mkdir(exist_ok=True)
fig = mm.pl.plot_intensity(mdata, modality="protein", groupby="factor value[condition]")
assert len(fig.data) > 0
fig.write_html(output_dir / "protein_intensities.html", include_plotlyjs=True)
fig.show(renderer="notebook_connected")


## 5. Save, reload, and check the result

The `.h5mu` file carries quantification, annotations, and recorded processing history.
[`mm.read_h5mu()`](../../reference/read_h5mu/) preserves that history and adds a read event. The checks below verify
that protein values and sample metadata survive the round trip.


In [6]:
result_path = output_dir / "analysis.h5mu"
mdata.write_h5mu(result_path)
loaded = mm.read_h5mu(result_path)

np.testing.assert_allclose(loaded["protein"].X, mdata["protein"].X, equal_nan=True)
# HDF5 can change categorical string storage; compare metadata values and axes.
pd.testing.assert_frame_equal(
    loaded["protein"].obs.astype(object), mdata["protein"].obs.astype(object)
)
assert loaded["protein"].var_names.equals(mdata["protein"].var_names)
print(f"Saved {loaded['protein'].n_vars} protein groups to {result_path}")


Saved 1475 protein groups to msmu_quick_start_output/analysis.h5mu


For group comparisons with this dataset, continue with [DE Analysis](https://bertis-informatics.github.io/msmu/tutorials/dea/),
which uses `factor value[condition]`, control `G1`, and experiment `G2`.
See [Provenance](https://bertis-informatics.github.io/msmu/how-it-works/provenance/) for recording and replay limitations.

## Dataset reference

Uszkoreit et al. (2022), *Dataset containing physiological amounts of spike-in
proteins into murine C2C12 background as a ground truth quantitative LC-MS/MS
reference*, Data in Brief 43, 108435.
The sample is distributed with `msmu` under `data/sage_lfq` (PXD012986).
